# Visit With Us — Wellness Tourism Package Purchase Prediction

**End-to-end machine learning and MLOps project**

This notebook prepares a customer purchase prediction model and creates the supporting project files for GitHub Actions and Streamlit Community Cloud. The workflow follows the supplied learner template and uses the attached `tourism.csv` dataset.

**Target:** `ProdTaken` (1 = purchased, 0 = did not purchase).

## 1. Business Context and Objective

Visit with Us wants to identify customers who are more likely to purchase the Wellness Tourism Package. A repeatable machine learning workflow can support customer segmentation and help the marketing team prioritize outreach.

The notebook covers dataset registration, data preparation, model experimentation, evaluation, deployment assets, and a GitHub Actions workflow.

## 2. Data Description

The dataset contains customer profile fields and sales-interaction fields. The target column is `ProdTaken`.

**Feature timing:** the stated business objective is to score customers before contacting them. For that reason, the model will use customer profile attributes only. Contact method, pitch duration, follow-up count, product pitched, and pitch satisfaction are excluded because they are observed during or after a sales interaction. Identifier columns are also excluded.

## 3. Import Libraries and Set Up the Project

The notebook uses pandas, scikit-learn and joblib for data handling and modelling. MLflow is used for experiment tracking when available (it is included in the supplied template). If MLflow is unavailable in the current notebook runtime, the tuned parameters and metrics are saved to a CSV log instead. The GitHub Actions training environment installs MLflow through the project requirements file.

In [1]:
import os
import json
import shutil
import runpy
from pathlib import Path

import pandas as pd
import sklearn
import joblib

print("pandas version:", pd.__version__)
print("scikit-learn version:", sklearn.__version__)
print("Project setup imports completed.")

pandas version: 2.2.3
scikit-learn version: 1.8.0
Project setup imports completed.


In [2]:
# Locate the attached dataset or the copy already placed in the project data folder.
possible_dataset_paths = [
    Path("/mnt/data/tourism.csv"),
    Path("tourism.csv"),
    Path("data/tourism.csv"),
    Path("tourism_project/data/tourism.csv"),
]

dataset_source = next((path for path in possible_dataset_paths if path.exists()), None)
if dataset_source is None:
    raise FileNotFoundError(
        "tourism.csv was not found. Place the dataset beside this notebook "
        "or inside tourism_project/data/ and run this cell again."
    )

PROJECT_ROOT = Path("tourism_project")
DATA_DIR = PROJECT_ROOT / "data"
MODEL_BUILDING_DIR = PROJECT_ROOT / "model_building"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"
WORKFLOW_DIR = Path(".github") / "workflows"

for folder in [DATA_DIR, MODEL_BUILDING_DIR, DEPLOYMENT_DIR, WORKFLOW_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

dataset_path = DATA_DIR / "tourism.csv"
if dataset_source.resolve() != dataset_path.resolve():
    shutil.copy2(dataset_source, dataset_path)

print("Dataset source:", dataset_source)
print("Registered project dataset:", dataset_path)
print("Project folders created successfully.")

Dataset source: /mnt/data/tourism.csv
Registered project dataset: tourism_project/data/tourism.csv
Project folders created successfully.


## 4. Initial Data Inspection

The first inspection checks the dataset dimensions, column names, data types, missing values, duplicate records and target distribution. This helps identify data quality issues before model preparation.

In [3]:
data = pd.read_csv(dataset_path)

print("Dataset shape:", data.shape)
display(data.head())
print("\nColumn data types:")
display(data.dtypes.to_frame("Data Type"))
print("\nMissing values by column:")
display(data.isna().sum().to_frame("Missing Values"))
print("\nDuplicate records:", data.duplicated().sum())
print("\nTarget distribution:")
display(data["ProdTaken"].value_counts().sort_index().to_frame("Customer Count"))

target_share = (data["ProdTaken"].value_counts(normalize=True).sort_index() * 100).round(2)
print("\nTarget distribution (%):")
display(target_share.to_frame("Percentage"))

Dataset shape: (4128, 21)


,Unnamed: 0,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,5,200005,0,32.0,Company Invited,1,8.0,Salaried,Male,3,3.0,Basic,3.0,Single,1.0,0,5,1,1.0,Executive,18068.0



Column data types:


,Data Type
Unnamed: 0,int64
CustomerID,int64
ProdTaken,int64
Age,float64
TypeofContact,object
CityTier,int64
DurationOfPitch,float64
Occupation,object
Gender,object
NumberOfPersonVisiting,int64



Missing values by column:


,Missing Values
Unnamed: 0,0
CustomerID,0
ProdTaken,0
Age,0
TypeofContact,0
CityTier,0
DurationOfPitch,0
Occupation,0
Gender,0
NumberOfPersonVisiting,0



Duplicate records: 0

Target distribution:


,Customer Count
ProdTaken,
0,3331
1,797



Target distribution (%):


,Percentage
ProdTaken,
0,80.69
1,19.31


**Observation:** the target is imbalanced: customers who did not purchase are the larger group. The evaluation therefore includes precision, recall, F1-score and ROC-AUC in addition to accuracy. The source also contains an index-like column and a spelling variation in the Gender category; these are handled during preparation.

## 5. Data Registration

A validation script checks that the expected dataset columns are present and prints a summary. The CSV is stored in the repository's `tourism_project/data/` directory so the GitHub Actions workflow can load it directly.

In [4]:
data_register_source = '\n"""Validate and summarize the tourism customer dataset."""\nfrom pathlib import Path\nimport pandas as pd\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nDATA_PATH = PROJECT_ROOT / "data" / "tourism.csv"\n\nEXPECTED_COLUMNS = [\'Unnamed: 0\', \'CustomerID\', \'ProdTaken\', \'Age\', \'TypeofContact\', \'CityTier\', \'DurationOfPitch\', \'Occupation\', \'Gender\', \'NumberOfPersonVisiting\', \'NumberOfFollowups\', \'ProductPitched\', \'PreferredPropertyStar\', \'MaritalStatus\', \'NumberOfTrips\', \'Passport\', \'PitchSatisfactionScore\', \'OwnCar\', \'NumberOfChildrenVisiting\', \'Designation\', \'MonthlyIncome\']\n\ndef main():\n    if not DATA_PATH.exists():\n        raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")\n    data = pd.read_csv(DATA_PATH)\n    missing_columns = sorted(set(EXPECTED_COLUMNS) - set(data.columns))\n    if missing_columns:\n        raise ValueError(f"Dataset is missing expected columns: {missing_columns}")\n    if "ProdTaken" not in data.columns:\n        raise ValueError("Target column \'ProdTaken\' is required.")\n    print("Dataset registration successful.")\n    print(f"File: {DATA_PATH}")\n    print(f"Shape: {data.shape}")\n    print(f"Duplicate rows: {data.duplicated().sum()}")\n    print("Missing values by column:")\n    print(data.isna().sum().to_string())\n    print("Target distribution:")\n    print(data["ProdTaken"].value_counts(dropna=False).sort_index().to_string())\n    return data\n\nif __name__ == "__main__":\n    main()\n'
(MODEL_BUILDING_DIR / "data_register.py").write_text(data_register_source, encoding="utf-8")
print("Created:", MODEL_BUILDING_DIR / "data_register.py")

registration_output = runpy.run_path(
    str(MODEL_BUILDING_DIR / "data_register.py"),
    run_name="__main__"
)


Created: tourism_project/model_building/data_register.py
Dataset registration successful.
File: /mnt/data/tourism_project/data/tourism.csv
Shape: (4128, 21)
Duplicate rows: 0
Missing values by column:
Unnamed: 0                  0
CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
Target distribution:
ProdTaken
0    3331
1     797


## 6. Data Preparation

The preparation script removes duplicate records and identifier columns, standardizes the `Gender` category, and excludes fields that would only be known after a customer interaction. It then creates a stratified 80:20 train/test split and saves the four split files for the next pipeline job.

In [5]:
prep_source = '\n"""Clean the dataset and create reproducible train/test CSV files."""\nfrom pathlib import Path\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nDATA_PATH = PROJECT_ROOT / "data" / "tourism.csv"\nTARGET = "ProdTaken"\n\n# Use only customer profile information available before the sales interaction.\nFEATURES = [\'Age\', \'CityTier\', \'Occupation\', \'Gender\', \'NumberOfPersonVisiting\', \'PreferredPropertyStar\', \'MaritalStatus\', \'NumberOfTrips\', \'Passport\', \'OwnCar\', \'NumberOfChildrenVisiting\', \'Designation\', \'MonthlyIncome\']\nPOST_CONTACT_COLUMNS = [\n    "TypeofContact", "DurationOfPitch", "NumberOfFollowups",\n    "ProductPitched", "PitchSatisfactionScore"\n]\n\ndef prepare_data():\n    data = pd.read_csv(DATA_PATH)\n    data = data.drop_duplicates().copy()\n\n    # Remove source/index identifiers; they do not describe customer behavior.\n    data = data.drop(columns=["Unnamed: 0", "CustomerID"], errors="ignore")\n\n    # Standardize a category spelling found in the source data.\n    if "Gender" in data.columns:\n        data["Gender"] = data["Gender"].replace({"Fe Male": "Female"})\n\n    # Exclude sales-interaction fields to keep the model suitable for pre-contact scoring.\n    data = data.drop(columns=POST_CONTACT_COLUMNS, errors="ignore")\n\n    if TARGET not in data.columns:\n        raise ValueError(f"Target column \'{TARGET}\' is missing.")\n    if data[TARGET].isna().any():\n        raise ValueError("Target column contains missing values.")\n    data[TARGET] = data[TARGET].astype(int)\n\n    X = data[FEATURES].copy()\n    y = data[TARGET].copy()\n\n    X_train, X_test, y_train, y_test = train_test_split(\n        X, y, test_size=0.20, random_state=42, stratify=y\n    )\n\n    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)\n    X_train.to_csv(PROJECT_ROOT / "Xtrain.csv", index=False)\n    X_test.to_csv(PROJECT_ROOT / "Xtest.csv", index=False)\n    y_train.to_frame(name=TARGET).to_csv(PROJECT_ROOT / "ytrain.csv", index=False)\n    y_test.to_frame(name=TARGET).to_csv(PROJECT_ROOT / "ytest.csv", index=False)\n\n    print(f"Training features: {X_train.shape}")\n    print(f"Testing features: {X_test.shape}")\n    print(f"Training target distribution: {y_train.value_counts().to_dict()}")\n    print(f"Testing target distribution: {y_test.value_counts().to_dict()}")\n    print("Train/test CSV files saved in:", PROJECT_ROOT)\n    return X_train, X_test, y_train, y_test\n\nif __name__ == "__main__":\n    prepare_data()\n'
(MODEL_BUILDING_DIR / "prep.py").write_text(prep_source, encoding="utf-8")
print("Created:", MODEL_BUILDING_DIR / "prep.py")

prep_output = runpy.run_path(
    str(MODEL_BUILDING_DIR / "prep.py"),
    run_name="__main__"
)

X_train = pd.read_csv(PROJECT_ROOT / "Xtrain.csv")
X_test = pd.read_csv(PROJECT_ROOT / "Xtest.csv")
y_train = pd.read_csv(PROJECT_ROOT / "ytrain.csv")["ProdTaken"]
y_test = pd.read_csv(PROJECT_ROOT / "ytest.csv")["ProdTaken"]

print("\nTraining feature sample:")
display(X_train.head())
print("\nFeature columns used by the model:")
print(X_train.columns.tolist())


Created: tourism_project/model_building/prep.py
Training features: (3302, 13)
Testing features: (826, 13)
Training target distribution: {0: 2664, 1: 638}
Testing target distribution: {0: 667, 1: 159}
Train/test CSV files saved in: /mnt/data/tourism_project

Training feature sample:


,Age,CityTier,Occupation,Gender,NumberOfPersonVisiting,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,55.0,1,Small Business,Female,4,5.0,Unmarried,8.0,1,0,1.0,Manager,23118.0
1,39.0,1,Salaried,Male,3,3.0,Unmarried,7.0,1,0,2.0,Executive,22622.0
2,42.0,2,Small Business,Male,3,5.0,Divorced,1.0,0,0,2.0,Manager,21272.0
3,37.0,1,Salaried,Female,3,5.0,Divorced,2.0,1,1,1.0,Executive,98678.0
4,23.0,1,Salaried,Male,3,3.0,Divorced,8.0,0,1,1.0,Manager,23453.0



Feature columns used by the model:
['Age', 'CityTier', 'Occupation', 'Gender', 'NumberOfPersonVisiting', 'PreferredPropertyStar', 'MaritalStatus', 'NumberOfTrips', 'Passport', 'OwnCar', 'NumberOfChildrenVisiting', 'Designation', 'MonthlyIncome']


**Observation:** stratification preserves the purchase/non-purchase proportions in both splits. The final model uses customer profile attributes only, which better reflects the intended pre-contact use case. Missing numeric and categorical values are handled inside the model pipeline, so preprocessing is fitted on training data and then consistently applied to test data.

## 7. Model Building, Hyperparameter Tuning and Experiment Tracking

A Random Forest classifier is used with a preprocessing pipeline. Numeric fields are imputed with the median, categorical fields are imputed with the most frequent value and one-hot encoded. `GridSearchCV` tests the specified hyperparameter combinations using stratified three-fold cross-validation and F1-score as the selection metric.

The best model, tuned parameters, test metrics and confusion matrix are saved under `tourism_project/deployment/`.

In [6]:
train_source = '\n"""Tune, evaluate, track and save the customer purchase prediction model."""\nfrom pathlib import Path\nimport json\nimport pandas as pd\nimport joblib\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.model_selection import GridSearchCV\nfrom sklearn.metrics import (\n    accuracy_score, precision_score, recall_score, f1_score,\n    roc_auc_score, classification_report, confusion_matrix\n)\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nTARGET = "ProdTaken"\nFEATURES = [\'Age\', \'CityTier\', \'Occupation\', \'Gender\', \'NumberOfPersonVisiting\', \'PreferredPropertyStar\', \'MaritalStatus\', \'NumberOfTrips\', \'Passport\', \'OwnCar\', \'NumberOfChildrenVisiting\', \'Designation\', \'MonthlyIncome\']\nCATEGORICAL_FEATURES = [\'Occupation\', \'Gender\', \'MaritalStatus\', \'Designation\']\nNUMERIC_FEATURES = [col for col in FEATURES if col not in CATEGORICAL_FEATURES]\n\ndef train_and_evaluate():\n    X_train = pd.read_csv(PROJECT_ROOT / "Xtrain.csv")\n    X_test = pd.read_csv(PROJECT_ROOT / "Xtest.csv")\n    y_train = pd.read_csv(PROJECT_ROOT / "ytrain.csv")[TARGET]\n    y_test = pd.read_csv(PROJECT_ROOT / "ytest.csv")[TARGET]\n\n    numeric_pipeline = Pipeline(steps=[\n        ("imputer", SimpleImputer(strategy="median"))\n    ])\n    categorical_pipeline = Pipeline(steps=[\n        ("imputer", SimpleImputer(strategy="most_frequent")),\n        ("encoder", OneHotEncoder(handle_unknown="ignore"))\n    ])\n    preprocessor = ColumnTransformer(transformers=[\n        ("numeric", numeric_pipeline, NUMERIC_FEATURES),\n        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES)\n    ])\n\n    pipeline = Pipeline(steps=[\n        ("preprocessor", preprocessor),\n        ("classifier", RandomForestClassifier(\n            random_state=42, class_weight="balanced", n_jobs=1\n        ))\n    ])\n\n    parameter_grid = {\n        "classifier__n_estimators": [150],\n        "classifier__max_depth": [None, 12],\n        "classifier__min_samples_leaf": [1, 3]\n    }\n    search = GridSearchCV(\n        estimator=pipeline,\n        param_grid=parameter_grid,\n        scoring="f1",\n        cv=3,\n        n_jobs=1,\n        refit=True,\n        verbose=1\n    )\n    search.fit(X_train, y_train)\n    best_model = search.best_estimator_\n\n    # Keep the results for every tested parameter combination.\n    grid_results = pd.DataFrame(search.cv_results_)[\n        ["params", "mean_test_score", "std_test_score", "rank_test_score"]\n    ].sort_values("rank_test_score")\n    grid_results_path = PROJECT_ROOT / "deployment" / "grid_search_results.csv"\n    grid_results_path.parent.mkdir(parents=True, exist_ok=True)\n    grid_results.to_csv(grid_results_path, index=False)\n\n    predictions = best_model.predict(X_test)\n    probabilities = best_model.predict_proba(X_test)[:, 1]\n    metrics = {\n        "accuracy": float(accuracy_score(y_test, predictions)),\n        "precision": float(precision_score(y_test, predictions, zero_division=0)),\n        "recall": float(recall_score(y_test, predictions, zero_division=0)),\n        "f1_score": float(f1_score(y_test, predictions, zero_division=0)),\n        "roc_auc": float(roc_auc_score(y_test, probabilities))\n    }\n\n    model_dir = PROJECT_ROOT / "deployment"\n    model_dir.mkdir(parents=True, exist_ok=True)\n    model_path = model_dir / "best_model.pkl"\n    joblib.dump(best_model, model_path)\n\n    results = {\n        "best_params": search.best_params_,\n        "best_cv_f1": float(search.best_score_),\n        "test_metrics": metrics,\n        "confusion_matrix": confusion_matrix(y_test, predictions).tolist()\n    }\n    (model_dir / "training_results.json").write_text(\n        json.dumps(results, indent=4), encoding="utf-8"\n    )\n\n    # MLflow is part of the supplied template. If it is not installed in a\n    # local environment, keep a CSV experiment log so the notebook still runs.\n    try:\n        import mlflow\n        mlflow.set_tracking_uri("file:" + str(PROJECT_ROOT / "mlruns"))\n        mlflow.set_experiment("Visit_With_Us_Wellness_Tourism")\n        with mlflow.start_run():\n            mlflow.log_params(search.best_params_)\n            mlflow.log_param("model", "RandomForestClassifier")\n            mlflow.log_param("cv_strategy", "Stratified 3-fold")\n            mlflow.log_param("parameter_grid", json.dumps(parameter_grid))\n            mlflow.log_metrics(metrics)\n            mlflow.log_metric("best_cv_f1", float(search.best_score_))\n            mlflow.log_artifact(str(model_path))\n            mlflow.log_artifact(str(grid_results_path))\n        print("Experiment details logged to MLflow.")\n    except ImportError:\n        log_row = {**search.best_params_, **metrics, "best_cv_f1": float(search.best_score_)}\n        pd.DataFrame([log_row]).to_csv(\n            PROJECT_ROOT / "deployment" / "experiment_log.csv", index=False\n        )\n        print("MLflow is not installed in this runtime; parameters, metrics and all grid-search results were saved locally.")\n\n    print("\\nBest parameters:", search.best_params_)\n    print("Best cross-validation F1:", round(search.best_score_, 4))\n    print("\\nTest metrics:")\n    print(pd.Series(metrics).round(4).to_string())\n    print("\\nClassification report:")\n    print(classification_report(y_test, predictions, zero_division=0))\n    print("Confusion matrix:")\n    print(confusion_matrix(y_test, predictions))\n    print("\\\\nSaved model:", model_path)\n    return best_model, metrics, results\n\nif __name__ == "__main__":\n    train_and_evaluate()\n'
(MODEL_BUILDING_DIR / "train.py").write_text(train_source, encoding="utf-8")
print("Created:", MODEL_BUILDING_DIR / "train.py")

training_output = runpy.run_path(
    str(MODEL_BUILDING_DIR / "train.py"),
    run_name="__main__"
)

results_path = DEPLOYMENT_DIR / "training_results.json"
training_results = json.loads(results_path.read_text(encoding="utf-8"))
print("\nSaved evaluation summary:")
display(pd.DataFrame([training_results["test_metrics"]]).T.rename(columns={0: "Test Set"}))
print("\nTuned parameters:")
display(pd.Series(training_results["best_params"], name="Best Value").to_frame())


Created: tourism_project/model_building/train.py


Fitting 3 folds for each of 4 candidates, totalling 12 fits


MLflow is not installed in this runtime; parameters, metrics and all grid-search results were saved locally.

Best parameters: {'classifier__max_depth': 12, 'classifier__min_samples_leaf': 3, 'classifier__n_estimators': 150}
Best cross-validation F1: 0.5974

Test metrics:
accuracy     0.8450
precision    0.6054
recall       0.5597
f1_score     0.5817
roc_auc      0.8912

Classification report:
              precision    recall  f1-score   support

           0       0.90      0.91      0.90       667
           1       0.61      0.56      0.58       159

    accuracy                           0.85       826
   macro avg       0.75      0.74      0.74       826
weighted avg       0.84      0.85      0.84       826

Confusion matrix:
[[609  58]
 [ 70  89]]
\nSaved model: /mnt/data/tourism_project/deployment/best_model.pkl

Saved evaluation summary:


,Test Set
accuracy,0.845036
precision,0.605442
recall,0.559748
f1_score,0.581699
roc_auc,0.891243



Tuned parameters:


,Best Value
classifier__max_depth,12
classifier__min_samples_leaf,3
classifier__n_estimators,150


### Model Evaluation

Accuracy gives the overall proportion of correct predictions. Precision indicates how many customers predicted as buyers actually purchased; recall indicates how many actual buyers were identified. F1-score balances precision and recall, while ROC-AUC evaluates ranking quality across classification thresholds.

The purchase class is less frequent, so the recall and F1-score for class 1 should be considered when deciding how the marketing team will use the predictions. The final threshold and campaign capacity should be selected with business stakeholders rather than relying on accuracy alone.

## 8. Model Deployment — Streamlit Application

The Streamlit app loads the saved model, collects the customer profile inputs into a DataFrame with the same feature names used during training, and displays the predicted class and purchase probability. The app and its deployment dependencies are written to `tourism_project/deployment/`.

In [7]:
app_source = 'import pandas as pd\nimport streamlit as st\nimport joblib\nfrom pathlib import Path\n\nst.set_page_config(page_title="Wellness Tourism Purchase Predictor", page_icon="🌿", layout="centered")\n\nMODEL_PATH = Path(__file__).resolve().parent / "best_model.pkl"\n\n@st.cache_resource\ndef load_model():\n    if not MODEL_PATH.exists():\n        raise FileNotFoundError(\n            "The trained model is not available yet. Run the GitHub Actions pipeline "\n            "and ensure best_model.pkl is committed to this folder."\n        )\n    return joblib.load(MODEL_PATH)\n\nst.title("Wellness Tourism Package")\nst.subheader("Customer Purchase Prediction")\nst.write(\n    "Enter the customer\'s profile details to estimate the likelihood of purchasing "\n    "the Wellness Tourism Package."\n)\n\nwith st.form("customer_details"):\n    age = st.number_input("Age", min_value=18, max_value=100, value=35, step=1)\n    city_tier = st.selectbox("City Tier", [1, 2, 3], index=0)\n    occupation = st.selectbox(\n        "Occupation", ["Salaried", "Small Business", "Large Business", "Free Lancer"]\n    )\n    gender = st.selectbox("Gender", ["Male", "Female"])\n    people_visiting = st.number_input(\n        "Number of People Visiting", min_value=1, max_value=10, value=2, step=1\n    )\n    preferred_star = st.selectbox("Preferred Property Star", [3, 4, 5], index=0)\n    marital_status = st.selectbox(\n        "Marital Status", ["Married", "Divorced", "Unmarried", "Single"]\n    )\n    trips = st.number_input("Number of Trips per Year", min_value=0, max_value=30, value=3)\n    passport = st.selectbox("Has Passport?", [0, 1], format_func=lambda x: "Yes" if x else "No")\n    own_car = st.selectbox("Owns a Car?", [0, 1], format_func=lambda x: "Yes" if x else "No")\n    children = st.number_input(\n        "Number of Children Visiting (below 5)", min_value=0, max_value=10, value=0\n    )\n    designation = st.selectbox(\n        "Designation", ["Executive", "Manager", "Senior Manager", "AVP", "VP"]\n    )\n    monthly_income = st.number_input(\n        "Monthly Income", min_value=0, max_value=200000, value=25000, step=1000\n    )\n    submitted = st.form_submit_button("Predict Purchase Likelihood")\n\nif submitted:\n    customer = pd.DataFrame([{\n        "Age": age,\n        "CityTier": city_tier,\n        "Occupation": occupation,\n        "Gender": gender,\n        "NumberOfPersonVisiting": people_visiting,\n        "PreferredPropertyStar": preferred_star,\n        "MaritalStatus": marital_status,\n        "NumberOfTrips": trips,\n        "Passport": passport,\n        "OwnCar": own_car,\n        "NumberOfChildrenVisiting": children,\n        "Designation": designation,\n        "MonthlyIncome": monthly_income\n    }])\n\n    try:\n        model = load_model()\n        prediction = int(model.predict(customer)[0])\n        probability = float(model.predict_proba(customer)[0][1])\n        st.metric("Estimated purchase probability", f"{probability:.1%}")\n        if prediction == 1:\n            st.success("The model identifies this customer as a potential buyer.")\n        else:\n            st.info("The model does not currently classify this customer as a likely buyer.")\n        st.caption("This is a model-based estimate, not a guarantee of customer behavior.")\n    except FileNotFoundError as error:\n        st.error(str(error))\n'
(DEPLOYMENT_DIR / "app.py").write_text(app_source, encoding="utf-8")

deployment_requirements = "streamlit==1.45.1\\npandas==2.2.3\\nscikit-learn==1.8.0\\njoblib==1.5.1\\n"
(DEPLOYMENT_DIR / "requirements.txt").write_text(deployment_requirements, encoding="utf-8")

print("Created deployment files:")
print(DEPLOYMENT_DIR / "app.py")
print(DEPLOYMENT_DIR / "requirements.txt")
print("Model file exists:", (DEPLOYMENT_DIR / "best_model.pkl").exists())


Created deployment files:
tourism_project/deployment/app.py
tourism_project/deployment/requirements.txt
Model file exists: True


### Streamlit Community Cloud Deployment

After the repository has been pushed and the GitHub Actions workflow has completed:

1. Sign in to [Streamlit Community Cloud](https://share.streamlit.io/) using the GitHub account that owns the repository.
2. Create an app from the public GitHub repository.
3. Select branch `main`.
4. Set the main file path to `tourism_project/deployment/app.py`.
5. Select Python 3.11 and deploy.
6. Confirm that the public app loads the committed `best_model.pkl` file.

The app URL should be added in the Output Evaluation section after deployment.

## 9. MLOps Pipeline with GitHub Actions

The workflow has three dependent jobs:

1. **Register dataset:** validates the dataset schema and prints a data summary.
2. **Prepare data:** creates the train/test CSV files and uploads them as a workflow artifact.
3. **Train and register model:** downloads the artifact, tunes and evaluates the model, then commits the updated model and evaluation summary to `main`.

The model commit includes `[skip ci]` to prevent the model update from triggering an unnecessary second training run.

In [8]:
workflow_source = 'name: Visit With Us - Tourism MLOps Pipeline\n\non:\n  push:\n    branches:\n      - main\n  workflow_dispatch:\n\npermissions:\n  contents: write\n\njobs:\n  register-dataset:\n    runs-on: ubuntu-latest\n    steps:\n      - name: Checkout repository\n        uses: actions/checkout@v4\n      - name: Set up Python\n        uses: actions/setup-python@v5\n        with:\n          python-version: "3.11"\n      - name: Install project dependencies\n        run: pip install -r requirements-ci.txt\n      - name: Validate and register dataset\n        run: python tourism_project/model_building/data_register.py\n\n  prepare-data:\n    needs: register-dataset\n    runs-on: ubuntu-latest\n    steps:\n      - name: Checkout repository\n        uses: actions/checkout@v4\n      - name: Set up Python\n        uses: actions/setup-python@v5\n        with:\n          python-version: "3.11"\n      - name: Install project dependencies\n        run: pip install -r requirements-ci.txt\n      - name: Prepare training and testing data\n        run: python tourism_project/model_building/prep.py\n      - name: Share train/test data with the next job\n        uses: actions/upload-artifact@v4\n        with:\n          name: tourism-data-splits\n          path: |\n            tourism_project/Xtrain.csv\n            tourism_project/Xtest.csv\n            tourism_project/ytrain.csv\n            tourism_project/ytest.csv\n          if-no-files-found: error\n          retention-days: 1\n\n  train-and-register-model:\n    needs: prepare-data\n    runs-on: ubuntu-latest\n    steps:\n      - name: Checkout repository\n        uses: actions/checkout@v4\n      - name: Set up Python\n        uses: actions/setup-python@v5\n        with:\n          python-version: "3.11"\n      - name: Install project dependencies\n        run: pip install -r requirements-ci.txt\n      - name: Download train/test artifact\n        uses: actions/download-artifact@v4\n        with:\n          name: tourism-data-splits\n          path: tourism_project\n      - name: Train, tune and evaluate model\n        run: python tourism_project/model_building/train.py\n      - name: Commit updated model to main\n        if: github.ref == \'refs/heads/main\' && !contains(github.event.head_commit.message, \'[skip ci]\')\n        run: |\n          git config user.name "github-actions[bot]"\n          git config user.email "41898282+github-actions[bot]@users.noreply.github.com"\n          git add tourism_project/deployment/best_model.pkl tourism_project/deployment/training_results.json tourism_project/deployment/grid_search_results.csv\n          git diff --cached --quiet || (git commit -m "Update trained tourism model [skip ci]" && git push)\n'
WORKFLOW_DIR.mkdir(parents=True, exist_ok=True)
(WORKFLOW_DIR / "pipeline.yml").write_text(workflow_source, encoding="utf-8")

ci_requirements = "pandas==2.2.3\\nscikit-learn==1.8.0\\njoblib==1.5.1\\nmlflow==3.0.1\\n"
Path("requirements-ci.txt").write_text(ci_requirements, encoding="utf-8")

print("Created workflow:", WORKFLOW_DIR / "pipeline.yml")
print("Created CI dependencies:", Path("requirements-ci.txt"))

project_readme = """# Visit With Us — Wellness Tourism Purchase Prediction

## Run locally
```bash
pip install -r requirements-ci.txt
python tourism_project/model_building/data_register.py
python tourism_project/model_building/prep.py
python tourism_project/model_building/train.py
```

## Deploy the Streamlit app
Push this repository to GitHub, wait for the Actions workflow to complete, then create a Streamlit Community Cloud app using branch `main` and main file path `tourism_project/deployment/app.py`. Select Python 3.11.

The model uses customer profile attributes available before contact. Sales-interaction fields and identifier columns are excluded to avoid target leakage.
"""
Path("README.md").write_text(project_readme, encoding="utf-8")
Path(".gitignore").write_text(
    "__pycache__/\\n*.pyc\\nmlruns/\\ntourism_project/Xtrain.csv\\ntourism_project/Xtest.csv\\ntourism_project/ytrain.csv\\ntourism_project/ytest.csv\\n",
    encoding="utf-8"
)
print("Created README.md and .gitignore.")
print("\nWorkflow preview:")
print((WORKFLOW_DIR / "pipeline.yml").read_text(encoding="utf-8"))


Created workflow: .github/workflows/pipeline.yml
Created CI dependencies: requirements-ci.txt
Created README.md and .gitignore.

Workflow preview:
name: Visit With Us - Tourism MLOps Pipeline

on:
  push:
    branches:
      - main
  workflow_dispatch:

permissions:
  contents: write

jobs:
  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout repository
        uses: actions/checkout@v4
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install project dependencies
        run: pip install -r requirements-ci.txt
      - name: Validate and register dataset
        run: python tourism_project/model_building/data_register.py

  prepare-data:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - name: Checkout repository
        uses: actions/checkout@v4
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"
  

## 10. GitHub Repository Setup and Publishing

Create a **public** GitHub repository and place the project files in it. The repository should contain the `tourism_project/` directory, `.github/workflows/pipeline.yml`, `requirements-ci.txt` and `README.md`.

For a notebook-based upload, the supplied template uses PyGithub. The optional cell below publishes the generated project files if `PyGithub` is installed and a GitHub token is available in the `GH_TOKEN` environment variable. Do not write a token directly in this notebook. In Google Colab, use Secrets to provide `GH_TOKEN` securely.

If the optional publishing cell is skipped, push the generated project folder to GitHub using your normal Git workflow.

In [9]:
# Optional: publish project files using the PyGithub package already referenced
# in the supplied template. Set GH_TOKEN as an environment variable first.
github_token = os.environ.get("GH_TOKEN")

if not github_token:
    print("GitHub publishing skipped: GH_TOKEN is not set.")
    print("Create a public repository and push the generated project files before deployment.")
else:
    try:
        from github import Github, GithubException

        github_username = os.environ.get("GITHUB_USERNAME", "").strip()
        repository_name = os.environ.get("REPO_NAME", "visit-with-us-wellness-tourism").strip()
        github = Github(github_token)
        github_user = github.get_user()
        if github_username and github_username != github_user.login:
            raise ValueError("GITHUB_USERNAME does not match the account associated with GH_TOKEN.")

        try:
            repository = github_user.get_repo(repository_name)
            print("Using existing repository:", repository.full_name)
        except GithubException as error:
            if getattr(error, "status", None) != 404:
                raise
            repository = github_user.create_repo(
                repository_name, private=False, auto_init=True,
                description="Wellness Tourism Package purchase prediction MLOps project"
            )
            print("Created public repository:", repository.full_name)

        upload_roots = [PROJECT_ROOT, Path("requirements-ci.txt"), Path("README.md"), Path(".gitignore"), Path(".github")]
        upload_files = []
        for item in upload_roots:
            if item.is_file():
                upload_files.append(item)
            elif item.is_dir():
                upload_files.extend(path for path in item.rglob("*") if path.is_file())

        # Push the workflow folder last so all code and dependency files are present
        # before the first workflow-triggering commit is created.
        upload_files = [
            path for path in upload_files
            if path.name not in {"Xtrain.csv", "Xtest.csv", "ytrain.csv", "ytest.csv"}
            and "mlruns" not in path.parts
        ]
        for local_file in upload_files:
            if str(local_file).startswith(str(PROJECT_ROOT)):
                remote_path = str(local_file).replace("\\\\", "/")
            else:
                remote_path = str(local_file).replace("\\\\", "/")
            content = local_file.read_bytes()
            try:
                existing = repository.get_contents(remote_path)
                repository.update_file(
                    remote_path, "Update " + remote_path, content, existing.sha, branch="main"
                )
            except GithubException as error:
                if getattr(error, "status", None) != 404:
                    raise
                repository.create_file(
                    remote_path, "Add " + remote_path, content, branch="main"
                )

        print("Project files uploaded to:", repository.html_url)
        print("GitHub Actions will run after the workflow file is pushed.")
    except ImportError:
        print("PyGithub is not installed. Install the same PyGithub version used in the supplied template to use this optional cell.")


GitHub publishing skipped: GH_TOKEN is not set.
Create a public repository and push the generated project files before deployment.


## 11. Output Evaluation

Complete the details below after the GitHub workflow and Streamlit deployment have run successfully.

- **GitHub repository:** add the public repository URL.
- **GitHub Actions:** attach a screenshot showing a successful run of all workflow jobs.
- **Repository structure:** attach a screenshot showing the project folders and key files.
- **Streamlit Community Cloud:** add the public app URL.
- **Deployed application:** attach a screenshot showing the running prediction form and a sample prediction.

Do not enter a deployment URL until the corresponding service has actually been deployed.

In [10]:
# Add the real URLs after deployment. These are intentionally left blank until then.
GITHUB_REPOSITORY_URL = ""
STREAMLIT_APP_URL = ""

print("GitHub repository URL:", GITHUB_REPOSITORY_URL or "Add after repository creation")
print("Streamlit app URL:", STREAMLIT_APP_URL or "Add after successful deployment")
print("Evidence to attach: repository structure, successful Actions run, deployed app.")


GitHub repository URL: Add after repository creation
Streamlit app URL: Add after successful deployment
Evidence to attach: repository structure, successful Actions run, deployed app.


## 12. Summary and Business Insights

- The dataset contains 4,128 customer records and the target is imbalanced, with fewer package purchasers than non-purchasers.
- The data preparation process removes identifiers, standardizes the Gender category and preserves the target distribution through stratified splitting.
- Customer profile features are used for pre-contact scoring. Sales-interaction variables are excluded because they would not be available at the intended prediction point.
- The preprocessing and Random Forest model are combined in one scikit-learn pipeline so that the same transformations are applied consistently during training and deployment.
- Hyperparameter tuning uses cross-validation and F1-score. Test-set precision, recall, F1-score and ROC-AUC provide a more informative evaluation than accuracy alone.
- GitHub Actions automates dataset validation, data preparation, artifact transfer, model training and model registration.
- The Streamlit application provides a simple interface for estimating customer purchase likelihood.

**Business recommendation:** use the model as a decision-support tool to prioritize customer outreach, not as an automatic final decision. Monitor performance after deployment and retrain when new, representative customer outcomes become available.

## 13. Final Project Files

The notebook has generated the following main files:

- `tourism_project/data/tourism.csv`
- `tourism_project/model_building/data_register.py`
- `tourism_project/model_building/prep.py`
- `tourism_project/model_building/train.py`
- `tourism_project/deployment/best_model.pkl`
- `tourism_project/deployment/app.py`
- `tourism_project/deployment/requirements.txt`
- `.github/workflows/pipeline.yml`
- `requirements-ci.txt`
- `README.md`

Run the notebook from top to bottom before exporting the final HTML submission.